In [1]:
from typing import Any, TypedDict


class AgentState(TypedDict, total=False):
    messages: list[dict[str, Any]]
    user_request: str

    route: str
    route_reason: str
    missing_information: list[str]

    retrieved_documents: list[dict[str, Any]]

    selected_tool: str | None
    tool_arguments: dict[str, Any] | None
    tool_result: Any

    validation_status: str | None
    validation_errors: list[str]

    execution_status: str | None

    guardrail_result: str | None

    escalation_required: bool
    escalation_reason: str | None

    final_response: str | None
    output_guard_result: str | None

In [7]:
from pydantic import BaseModel, Field
from typing import Any

# Support both package execution and running this code directly in a notebook.
try:
    from .routing import Route
except ImportError:
    try:
        from routing import Route
    except ImportError:
        # The routing module is not available in this notebook context.
        # Use str as a permissive fallback so the models remain executable.
        Route = str


class RouteDecision(BaseModel):
    route: Route = Field(
        description="The high-level route for the user request."
    )

    reason: str = Field(
        description="A concise explanation for the routing decision."
    )

    missing_information: list[str] = Field(
        default_factory=list,
        description=(
            "Information that is required to proceed but is missing "
            "from the user's request. Empty when nothing is missing."
        ),
    )


class ToolCallDecision(BaseModel):
    tool_name: str = Field(
        description="The name of the tool to call."
    )

    arguments: dict[str, Any] = Field(
        description="Arguments to pass to the selected tool."
    )

    reason: str = Field(
        description="A concise explanation for selecting this tool."
    )


class ToolValidationResult(BaseModel):
    valid: bool = Field(
        description="Whether the tool call is valid."
    )

    errors: list[str] = Field(
        default_factory=list,
        description="Validation errors, if any."
    )

In [11]:
from pydantic import BaseModel, Field
from typing import Any

#from .routing import Route


class RouteDecision(BaseModel):
    route: Route = Field(
        description="The high-level route for the user request."
    )

    reason: str = Field(
        description="A concise explanation for the routing decision."
    )

    missing_information: list[str] = Field(
        default_factory=list,
        description=(
            "Information that is required to proceed but is missing "
            "from the user's request. Empty when nothing is missing."
        ),
    )


class ToolCallDecision(BaseModel):
    tool_name: str = Field(
        description="The name of the tool to call."
    )

    arguments: dict[str, Any] = Field(
        description="Arguments to pass to the selected tool."
    )

    reason: str = Field(
        description="A concise explanation for selecting this tool."
    )

class ToolValidationResult(BaseModel):
    valid: bool = Field(
        description="Whether the tool call is valid."
    )

    errors: list[str] = Field(
        default_factory=list,
        description="Validation errors, if any."
    )

In [15]:
import os
import sys
import subprocess

# Install the missing LangChain integration (and its dependencies) in this
# notebook's active Python environment, then import it.
try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
except ModuleNotFoundError as exc:
    if exc.name not in {"langchain_openai", "langchain_core", "openai"}:
        raise
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "langchain-openai"]
    )
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate

# RouteDecision and ToolCallDecision are defined in the schema cell earlier
# in this notebook. Import them when this code is run as part of a package.
try:
    from .schemas import RouteDecision, ToolCallDecision
except (ImportError, ValueError):
    try:
        from schemas import RouteDecision, ToolCallDecision
    except ImportError:
        # In a notebook, the schema definitions may already exist in globals.
        if "RouteDecision" not in globals() or "ToolCallDecision" not in globals():
            raise ImportError(
                "RouteDecision and ToolCallDecision are not defined. "
                "Run the schema-definition cell before this cell."
            )


ROUTER_SYSTEM_PROMPT = """
You are the routing component of an e-commerce customer support agent.

Your task is to determine the next route for the user's request.

Available routes:

- direct:
  Use when the request can be answered directly without retrieving
  external knowledge, using a tool, asking for clarification, or
  escalating to a human.

- rag:
  Use when the request requires static knowledge such as return policies,
  shipping policies, warranties, FAQs, or product information.

- tool:
  Use when the request requires dynamic data or an action such as checking
  an order, cancelling an order, initiating a return, updating an account,
  modifying a cart, or searching products.

- clarification:
  Use when the user's intent is understandable but required information is
  missing and must be requested before proceeding.

- human:
  Use when the request requires human intervention, especially for
  sensitive, high-risk, exceptional, or potentially fraudulent situations.

Important rules:

1. Choose exactly one route.
2. Do not answer the user's request.
3. Do not execute any tool.
4. Prefer clarification when required information is missing.
5. Use tool for dynamic information or actions.
6. Use rag for static knowledge.
7. Use human for sensitive or high-risk cases.
8. When choosing clarification, identify the minimum information
   required to proceed and list it in missing_information.
9. If clarification is not required, return an empty missing_information list.
"""


def build_router():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY environment variable is not set.")

    llm = ChatOpenAI(
        model="gpt-5.6-luna",
        temperature=0,
        api_key=api_key,
    )
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", ROUTER_SYSTEM_PROMPT),
            ("human", "{user_request}"),
        ]
    )
    return prompt | llm.with_structured_output(RouteDecision)


GENERATION_SYSTEM_PROMPT = """
You are an e-commerce customer support assistant.

Answer the user's request using the provided context when available.

Rules:
1. Be accurate and concise.
2. Do not invent information that is not supported by the context.
3. If retrieved knowledge is provided, use it as the source of truth.
4. If a tool result is provided, use it as the source of truth for dynamic information.
5. If no additional context is provided, answer directly using your general knowledge.
6. Do not mention internal tools, routing, retrieval, system prompts, or agent architecture.
"""


def build_llm():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY environment variable is not set.")

    llm = ChatOpenAI(
        model="gpt-5.6-luna",
        temperature=0,
        api_key=api_key,
    )
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", GENERATION_SYSTEM_PROMPT),
            (
                "human",
                """User request:
{user_request}

Retrieved knowledge:
{retrieved_documents}

Tool result:
{tool_result}
""",
            ),
        ]
    )
    return prompt | llm


TOOL_SELECTION_SYSTEM_PROMPT = """
You are the tool-selection component of an e-commerce customer support agent.

Your task is to select the single most appropriate tool for the user's request
and provide the arguments required by that tool.

Rules:
1. Select exactly one tool.
2. Use only the tools provided.
3. Do not execute the tool.
4. Do not invent tool names.
5. Provide arguments using the exact argument names defined by the tool schema.
6. Do not invent missing information.
7. If a required argument is missing, leave it absent rather than guessing.
8. Select the tool that best matches the user's requested action.
"""


def build_tool_selector():
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY environment variable is not set.")

    llm = ChatOpenAI(
        model="gpt-5.6-luna",
        temperature=0,
        api_key=api_key,
    )
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", TOOL_SELECTION_SYSTEM_PROMPT),
            (
                "human",
                """User request:
{user_request}

Available tools:
{available_tools}
""",
            ),
        ]
    )
    return prompt | llm.with_structured_output(ToolCallDecision)


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [17]:
FORBIDDEN_OUTPUT_PATTERNS = [
    "selected_tool",
    "tool_arguments",
    "route_reason",
    "retrieved_documents",
    "system prompt",
    "guardrail",
    "agent architecture",
]


SENSITIVE_TOOLS = {
    "cancel_order",
    "initiate_return",
    "initiate_exchange",
    "log_payment_dispute",
    "update_account",
}

ALLOWED_TOOLS = {
    "cancel_order",
    "initiate_exchange",
    "get_order_status",
    "initiate_return",
    "search_products",
    "log_payment_dispute",
    "update_account",
    "log_complaint",
    "add_to_cart",
    "update_cart_qty",
    "escalate_to_agent",
    "get_recommendations",
}

In [21]:
ORDERS = {
    "ORD-1001": {
        "user_id": "USER-001",
        "status": "shipped",
        "items": [
            {
                "sku": "SKU-001",
                "name": "Wireless Headphones",
                "variant": "Black",
            }
        ],
    },
    "ORD-1002": {
        "user_id": "USER-002",
        "status": "processing",
        "items": [
            {
                "sku": "SKU-002",
                "name": "Running Shoes",
                "variant": "Size 42",
            }
        ],
    },
}


PRODUCTS = [
    {
        "sku": "SKU-001",
        "name": "Wireless Headphones",
        "category": "electronics",
        "price": 79.99,
        "variants": ["Black", "White"],
    },
    {
        "sku": "SKU-002",
        "name": "Running Shoes",
        "category": "sports",
        "price": 64.99,
        "variants": ["Size 41", "Size 42", "Size 43"],
    },
]


CARTS = {
    "USER-001": [],
    "USER-002": [],
}


ACCOUNTS = {
    "USER-001": {
        "email": "user001@example.com",
        "phone": "+10000000001",
        "shipping_address": "123 Main Street",
    },
    "USER-002": {
        "email": "user002@example.com",
        "phone": "+10000000002",
        "shipping_address": "456 Oak Street",
    },
}

In [23]:
#from .data import ACCOUNTS


ALLOWED_ACCOUNT_FIELDS = {
    "shipping_address",
    "email",
    "phone",
    "password",
}


def update_account(
    user_id: str,
    field: str,
    value: str,
) -> dict:

    if field not in ALLOWED_ACCOUNT_FIELDS:
        return {
            "success": False,
            "error": "Unsupported account field.",
        }

    if user_id not in ACCOUNTS:
        return {
            "success": False,
            "error": "User not found.",
        }

    ACCOUNTS[user_id][field] = value

    return {
        "success": True,
        "user_id": user_id,
        "updated_field": field,
    }

In [25]:
#from .data import CARTS


def add_to_cart(
    user_id: str,
    item_sku: str,
    quantity: int,
) -> dict:

    CARTS.setdefault(user_id, [])

    CARTS[user_id].append(
        {
            "item_sku": item_sku,
            "quantity": quantity,
        }
    )

    return {
        "success": True,
        "user_id": user_id,
        "cart": CARTS[user_id],
    }


def update_cart_qty(
    user_id: str,
    item_sku: str,
    quantity: int,
) -> dict:

    cart = CARTS.get(user_id, [])

    for item in cart:
        if item["item_sku"] == item_sku:
            item["quantity"] = quantity

            return {
                "success": True,
                "user_id": user_id,
                "cart": cart,
            }

    return {
        "success": False,
        "error": "Item not found in cart.",
    }

In [27]:
#from .data import ORDERS


def get_order_status(
    order_id: str,
    user_id: str,
) -> dict:

    order = ORDERS.get(order_id)

    if order is None:
        return {
            "success": False,
            "error": "Order not found.",
        }

    if order["user_id"] != user_id:
        return {
            "success": False,
            "error": "Order does not belong to this user.",
        }

    return {
        "success": True,
        "order_id": order_id,
        "status": order["status"],
    }


def cancel_order(
    order_id: str,
    reason: str | None = None,
) -> dict:

    order = ORDERS.get(order_id)

    if order is None:
        return {
            "success": False,
            "error": "Order not found.",
        }

    if order["status"] in {"shipped", "delivered"}:
        return {
            "success": False,
            "error": "Order cannot be cancelled at this stage.",
        }

    order["status"] = "cancelled"

    return {
        "success": True,
        "order_id": order_id,
        "status": "cancelled",
        "reason": reason,
    }


def initiate_return(
    order_id: str,
    reason: str,
    item_sku: str | None = None,
) -> dict:

    order = ORDERS.get(order_id)

    if order is None:
        return {
            "success": False,
            "error": "Order not found.",
        }

    return {
        "success": True,
        "order_id": order_id,
        "item_sku": item_sku,
        "reason": reason,
        "status": "return_requested",
    }


def initiate_exchange(
    order_id: str,
    item_sku: str,
    new_variant: str,
) -> dict:

    order = ORDERS.get(order_id)

    if order is None:
        return {
            "success": False,
            "error": "Order not found.",
        }

    return {
        "success": True,
        "order_id": order_id,
        "item_sku": item_sku,
        "new_variant": new_variant,
        "status": "exchange_requested",
    }

In [29]:
#from .data import PRODUCTS


def search_products(
    query: str,
) -> dict:

    query = query.lower()

    matches = [
        product
        for product in PRODUCTS
        if query in product["name"].lower()
        or query in product["category"].lower()
    ]

    return {
        "success": True,
        "products": matches,
    }


def get_recommendations(
    user_id: str,
    category: str | None = None,
) -> dict:

    products = PRODUCTS

    if category:
        products = [
            product
            for product in products
            if product["category"] == category
        ]

    return {
        "success": True,
        "user_id": user_id,
        "recommendations": products[:3],
    }

In [31]:
def log_payment_dispute(
    user_id: str,
    transaction_id: str,
    reason: str,
) -> dict:

    return {
        "success": True,
        "user_id": user_id,
        "transaction_id": transaction_id,
        "status": "payment_dispute_logged",
        "reason": reason,
    }


def log_complaint(
    user_id: str,
    complaint: str,
) -> dict:

    return {
        "success": True,
        "user_id": user_id,
        "status": "complaint_logged",
        "complaint": complaint,
    }


def escalate_to_agent(
    reason: str,
    priority: str,
    context_summary: str | None = None,
) -> dict:

    return {
        "success": True,
        "status": "escalated",
        "priority": priority,
        "reason": reason,
        "context_summary": context_summary,
    }

In [33]:
TOOL_SCHEMAS = [
    {
        "name": "cancel_order",
        "description": "Cancel an order when cancellation is allowed.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order identifier.",
                },
                "reason": {
                    "type": "string",
                    "description": "Reason for cancellation.",
                },
            },
            "required": ["order_id"],
        },
    },
    {
        "name": "initiate_exchange",
        "description": "Initiate an exchange for an item in an order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order identifier.",
                },
                "item_sku": {
                    "type": "string",
                    "description": "SKU of the item to exchange.",
                },
                "new_variant": {
                    "type": "string",
                    "description": "The requested replacement variant.",
                },
            },
            "required": [
                "order_id",
                "item_sku",
                "new_variant",
            ],
        },
    },
    {
        "name": "get_order_status",
        "description": "Retrieve the current status of an order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order identifier.",
                },
                "user_id": {
                    "type": "string",
                    "description": "The customer identifier.",
                },
            },
            "required": [
                "order_id",
                "user_id",
            ],
        },
    },
    {
        "name": "initiate_return",
        "description": "Initiate a return for an order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order identifier.",
                },
                "reason": {
                    "type": "string",
                    "description": "Reason for the return.",
                },
                "item_sku": {
                    "type": "string",
                    "description": "Optional SKU of the returned item.",
                },
            },
            "required": [
                "order_id",
                "reason",
            ],
        },
    },
    {
        "name": "search_products",
        "description": "Search the product catalog.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Product search query.",
                },
            },
            "required": ["query"],
        },
    },
    {
        "name": "log_payment_dispute",
        "description": "Log a payment dispute for a customer.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The customer identifier.",
                },
                "transaction_id": {
                    "type": "string",
                    "description": "The transaction identifier.",
                },
                "reason": {
                    "type": "string",
                    "description": "Reason for the payment dispute.",
                },
            },
            "required": [
                "user_id",
                "transaction_id",
                "reason",
            ],
        },
    },
    {
        "name": "update_account",
        "description": "Update an allowed customer account field.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The customer identifier.",
                },
                "field": {
                    "type": "string",
                    "enum": [
                        "shipping_address",
                        "email",
                        "phone",
                        "password",
                    ],
                    "description": "Account field to update.",
                },
                "value": {
                    "type": "string",
                    "description": "New value for the account field.",
                },
            },
            "required": [
                "user_id",
                "field",
                "value",
            ],
        },
    },
    {
        "name": "log_complaint",
        "description": "Log a customer complaint.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The customer identifier.",
                },
                "complaint": {
                    "type": "string",
                    "description": "The complaint text.",
                },
            },
            "required": [
                "user_id",
                "complaint",
            ],
        },
    },
    {
        "name": "add_to_cart",
        "description": "Add a product to the customer's cart.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The customer identifier.",
                },
                "item_sku": {
                    "type": "string",
                    "description": "Product SKU.",
                },
                "quantity": {
                    "type": "integer",
                    "description": "Quantity to add.",
                },
            },
            "required": [
                "user_id",
                "item_sku",
                "quantity",
            ],
        },
    },
    {
        "name": "update_cart_qty",
        "description": "Update the quantity of an item in the cart.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The customer identifier.",
                },
                "item_sku": {
                    "type": "string",
                    "description": "Product SKU.",
                },
                "quantity": {
                    "type": "integer",
                    "description": "New quantity.",
                },
            },
            "required": [
                "user_id",
                "item_sku",
                "quantity",
            ],
        },
    },
    {
        "name": "escalate_to_agent",
        "description": "Escalate a customer issue to a human support agent.",
        "parameters": {
            "type": "object",
            "properties": {
                "reason": {
                    "type": "string",
                    "description": "Reason for escalation.",
                },
                "priority": {
                    "type": "string",
                    "description": "Escalation priority.",
                },
                "context_summary": {
                    "type": "string",
                    "description": "Optional summary of the issue.",
                },
            },
            "required": [
                "reason",
                "priority",
            ],
        },
    },
    {
        "name": "get_recommendations",
        "description": "Get product recommendations for a customer.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The customer identifier.",
                },
                "category": {
                    "type": "string",
                    "description": "Optional product category.",
                },
            },
            "required": ["user_id"],
        },
    },
]

In [39]:
from typing import Callable, Any

'''from .account_tools import update_account
from .cart_tools import add_to_cart, update_cart_qty
from .order_tools import (
    cancel_order,
    get_order_status,
    initiate_exchange,
    initiate_return,
)
from .product_tools import (
    get_recommendations,
    search_products,
)
from .support_tools import (
    escalate_to_agent,
    log_complaint,
    log_payment_dispute,
)
'''

ToolFunction = Callable[..., Any]


TOOL_REGISTRY: dict[str, ToolFunction] = {
    "cancel_order": cancel_order,
    "initiate_exchange": initiate_exchange,
    "get_order_status": get_order_status,
    "initiate_return": initiate_return,
    "search_products": search_products,
    "log_payment_dispute": log_payment_dispute,
    "update_account": update_account,
    "log_complaint": log_complaint,
    "add_to_cart": add_to_cart,
    "update_cart_qty": update_cart_qty,
    "escalate_to_agent": escalate_to_agent,
    "get_recommendations": get_recommendations,
}

In [41]:
from typing import Any

'''from ..tools.registry import ToolFunction
from .state import AgentState

from src.guardrails.policies import (
    ALLOWED_TOOLS,
    SENSITIVE_TOOLS,
)

from guardrails.output import FORBIDDEN_OUTPUT_PATTERNS
'''


def input_guard_node(state: AgentState) -> dict[str, Any]:
    """
    Validate the incoming user request before routing.

    This node performs lightweight deterministic checks.
    """

    user_request = state.get("user_request")

    if not isinstance(user_request, str):
        raise ValueError("user_request must be a string.")

    user_request = user_request.strip()

    if not user_request:
        raise ValueError("user_request cannot be empty.")

    return {
        "user_request": user_request,
    }

def router_node(
    state: AgentState,
    router,
) -> dict[str, Any]:

    user_request = state["user_request"]

    decision: RouteDecision = router.invoke(
        {
            "user_request": user_request
        }
    )

    return {
        "route": decision.route,
        "route_reason": decision.reason,
        "missing_information": decision.missing_information,
    }




def direct_node( state: AgentState, llm,) -> dict[str, Any]:

    user_request = state["user_request"]

    response = llm.invoke(
        user_request
    )

    return {
        "final_response": response.content,
    }


def retrieve_node(
    state: AgentState,
    retriever,
) -> dict[str, Any]:

    user_request = state["user_request"]

    documents = retriever.invoke(user_request)

    retrieved_documents = [
        {
            "content": document.page_content,
            "metadata": document.metadata,
        }
        for document in documents
    ]

    return {
        "retrieved_documents": retrieved_documents,
    }


def generate_response_node(
    state: AgentState,
    llm,
) -> dict[str, Any]:

    response = llm.invoke(
        {
            "user_request": state["user_request"],
            "retrieved_documents": state.get(
                "retrieved_documents",
                []
            ),
            "tool_result": state.get(
                "tool_result",
                None
            ),
        }
    )

    return {
        "final_response": response.content,
    }


def clarification_node(
    state: AgentState,
    llm,
) -> dict[str, Any]:

    user_request = state["user_request"]
    missing_information = state.get(
        "missing_information",
        [],
    )

    prompt = f"""
The user made the following request:

{user_request}

The following information is required to proceed:

{missing_information}

Ask the user for the minimum missing information needed.
Do not invent any information.
Be concise and natural.
"""

    response = llm.invoke(prompt)

    return {
        "final_response": response.content,
    }



def human_escalation_node(
    state: AgentState,
    llm,
) -> dict[str, Any]:

    user_request = state["user_request"]
    escalation_reason = state.get(
        "escalation_reason",
        "The request requires human assistance.",
    )

    prompt = f"""
The user made the following request:

{user_request}

The request must be escalated to a human support agent.

Reason for escalation:
{escalation_reason}

Write a concise and professional message informing the user
that their request will be handled by a human support agent.

Do not expose internal routing, guardrails, policies,
or agent architecture.
Do not claim that a human has already responded.
"""

    response = llm.invoke(prompt)

    return {
        "final_response": response.content,
        "escalation_required": True,
        "escalation_reason": escalation_reason,
    }


def tool_selection_node(
    state: AgentState,
    tool_selector,
    available_tools: list[dict[str, Any]],
) -> dict[str, Any]:

    user_request = state["user_request"]

    decision: ToolCallDecision = tool_selector.invoke(
        {
            "user_request": user_request,
            "available_tools": available_tools,
        }
    )

    return {
        "selected_tool": decision.tool_name,
        "tool_arguments": decision.arguments,
    }


def tool_validation_node(
    state: AgentState,
    available_tools: list[dict[str, Any]],
) -> dict[str, Any]:

    selected_tool = state.get("selected_tool")
    tool_arguments = state.get("tool_arguments")

    if not selected_tool:
        return {
            "validation_status": "invalid",
            "validation_errors": ["No tool was selected."],
        }

    if not isinstance(tool_arguments, dict):
        return {
            "validation_status": "invalid",
            "validation_errors": [
                "Tool arguments must be an object."
            ],
        }

    tool_schema = next(
        (
            tool
            for tool in available_tools
            if tool.get("name") == selected_tool
        ),
        None,
    )

    if tool_schema is None:
        return {
            "validation_status": "invalid",
            "validation_errors": [
                f"Unknown tool: {selected_tool}"
            ],
        }

    parameters = tool_schema.get("parameters", {})

    required_arguments = parameters.get(
        "required",
        [],
    )

    missing_arguments = [
        argument
        for argument in required_arguments
        if argument not in tool_arguments
    ]

    properties = parameters.get(
        "properties",
        {},
    )

    invalid_arguments = [
        argument
        for argument in tool_arguments
        if argument not in properties
    ]

    errors = []

    if missing_arguments:
        errors.extend(
            [
                f"Missing required argument: {argument}"
                for argument in missing_arguments
            ]
        )

    if invalid_arguments:
        errors.extend(
            [
                f"Unknown argument: {argument}"
                for argument in invalid_arguments
            ]
        )

    return {
        "validation_status": (
            "valid" if not errors else "invalid"
        ),
        "validation_errors": errors,
    }


def guardrail_node(
    state: AgentState,
) -> dict[str, Any]:

    validation_status = state.get(
        "validation_status"
    )

    selected_tool = state.get(
        "selected_tool"
    )

    if validation_status != "valid":
        return {
            "guardrail_result": "blocked",
            "escalation_required": True,
            "escalation_reason": (
                "Tool call failed validation."
            ),
        }

    if selected_tool not in ALLOWED_TOOLS:
        return {
            "guardrail_result": "blocked",
            "escalation_required": True,
            "escalation_reason": (
                f"Tool '{selected_tool}' is not allowed."
            ),
        }

    if selected_tool in SENSITIVE_TOOLS:
        return {
            "guardrail_result": "requires_review",
            "escalation_required": True,
            "escalation_reason": (
                f"Tool '{selected_tool}' requires "
                "human review."
            ),
        }

    return {
        "guardrail_result": "allowed",
        "escalation_required": False,
        "escalation_reason": None,
    }


def tool_execution_node(
    state: AgentState,
    tool_registry: dict[str, ToolFunction],
) -> dict[str, Any]:

    selected_tool = state.get("selected_tool")
    tool_arguments = state.get("tool_arguments", {})

    if not selected_tool:
        raise ValueError(
            "No tool selected for execution."
        )

    tool = tool_registry.get(selected_tool)

    if tool is None:
        raise ValueError(
            f"Tool '{selected_tool}' is not registered."
        )

    try:
        result = tool(**tool_arguments)

    except Exception as exc:
        return {
            "tool_result": None,
            "execution_status": "failed",
            "escalation_required": True,
            "escalation_reason": (
                f"Tool execution failed: {exc}"
            ),
        }

    return {
        "tool_result": result,
        "execution_status": "successful",
    }




def output_guard_node(
    state: AgentState,
) -> dict[str, Any]:

    final_response = state.get("final_response")

    if not isinstance(final_response, str):
        return {
            "output_guard_result": "blocked",
            "final_response": (
                "I’m sorry, but I’m unable to provide a response "
                "right now. Please contact a human support agent."
            ),
        }

    final_response = final_response.strip()

    if not final_response:
        return {
            "output_guard_result": "blocked",
            "final_response": (
                "I’m sorry, but I’m unable to provide a response "
                "right now. Please contact a human support agent."
            ),
        }

    response_lower = final_response.lower()

    for pattern in FORBIDDEN_OUTPUT_PATTERNS:
        if pattern.lower() in response_lower:
            return {
                "output_guard_result": "blocked",
                "final_response": (
                    "I’m sorry, but I’m unable to provide a response "
                    "right now. Please contact a human support agent."
                ),
            }

    if state.get("escalation_required"):
        misleading_phrases = [
            "a human agent has responded",
            "a human agent already responded",
            "i spoke with a human agent",
        ]

        if any(
            phrase in response_lower
            for phrase in misleading_phrases
        ):
            return {
                "output_guard_result": "blocked",
                "final_response": (
                    "Your request has been escalated to a human "
                    "support agent for further assistance."
                ),
            }

    return {
        "output_guard_result": "passed",
        "final_response": final_response,
    }

# 

# Graph

In [45]:
import sys
import subprocess

# Install LangGraph in the active notebook environment if it is not available.
try:
    from langgraph.graph import StateGraph, START, END
except ModuleNotFoundError as exc:
    if exc.name != "langgraph":
        raise
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "langgraph"]
    )
    from langgraph.graph import StateGraph, START, END

'''from .state import AgentState
from .nodes import (
    input_guard_node,
    router_node,
    retrieve_node,
    generate_response_node,
    clarification_node,
    human_escalation_node,
    tool_selection_node,
    tool_validation_node,
    guardrail_node,
    tool_execution_node,
    output_guard_node,
)
'''


def route_after_router(state: AgentState) -> str:
    route = state.get("route")

    if route not in {
        "direct",
        "rag",
        "tool",
        "clarification",
        "human",
    }:
        raise ValueError(f"Invalid route: {route}")

    return route


def route_after_validation(state: AgentState) -> str:
    validation_status = state.get("validation_status")

    if validation_status == "valid":
        return "valid"

    if validation_status == "invalid":
        return "invalid"

    raise ValueError(f"Invalid validation status: {validation_status}")


def route_after_guardrail(state: AgentState) -> str:
    result = state.get("guardrail_result")

    if result == "allowed":
        return "allowed"

    if result in {"requires_review", "blocked"}:
        return "human"

    raise ValueError(f"Invalid guardrail result: {result}")


def build_graph(
    router,
    llm,
    retriever,
    tool_selector,
    available_tools,
    tool_registry,
):
    graph = StateGraph(AgentState)

    # Nodes
    graph.add_node("input_guard", input_guard_node)
    graph.add_node(
        "router",
        lambda state: router_node(state, router),
    )
    graph.add_node(
        "direct",
        lambda state: generate_response_node(state, llm),
    )
    graph.add_node(
        "retrieve",
        lambda state: retrieve_node(state, retriever),
    )
    graph.add_node(
        "generate_response",
        lambda state: generate_response_node(state, llm),
    )
    graph.add_node(
        "clarification",
        lambda state: clarification_node(state, llm),
    )
    graph.add_node(
        "human_escalation",
        lambda state: human_escalation_node(state, llm),
    )
    graph.add_node(
        "tool_selection",
        lambda state: tool_selection_node(
            state,
            tool_selector,
            available_tools,
        ),
    )
    graph.add_node(
        "tool_validation",
        lambda state: tool_validation_node(state, available_tools),
    )
    graph.add_node("guardrail", guardrail_node)
    graph.add_node(
        "tool_execution",
        lambda state: tool_execution_node(state, tool_registry),
    )
    graph.add_node("output_guard", output_guard_node)

    # Entry and routing
    graph.add_edge(START, "input_guard")
    graph.add_edge("input_guard", "router")
    graph.add_conditional_edges(
        "router",
        route_after_router,
        {
            "direct": "direct",
            "rag": "retrieve",
            "tool": "tool_selection",
            "clarification": "clarification",
            "human": "human_escalation",
        },
    )

    # Direct and retrieval-based responses
    graph.add_edge("direct", "output_guard")
    graph.add_edge("retrieve", "generate_response")
    graph.add_edge("generate_response", "output_guard")

    # Clarification and human escalation
    graph.add_edge("clarification", "output_guard")
    graph.add_edge("human_escalation", "output_guard")

    # Tool workflow
    graph.add_edge("tool_selection", "tool_validation")
    graph.add_conditional_edges(
        "tool_validation",
        route_after_validation,
        {
            "valid": "guardrail",
            "invalid": "clarification",
        },
    )
    graph.add_conditional_edges(
        "guardrail",
        route_after_guardrail,
        {
            "allowed": "tool_execution",
            "human": "human_escalation",
        },
    )
    graph.add_edge("tool_execution", "generate_response")

    # Final output check
    graph.add_edge("output_guard", END)

    return graph.compile()


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [47]:
from typing import Any

'''from .graph import build_graph
from .llm import (
    build_llm,
    build_router,
    build_tool_selector,
)
from ..tools.registry import TOOL_REGISTRY
from ..tools.schemas import TOOL_SCHEMAS
'''

def build_agent(
    retriever: Any,
):
    router = build_router()
    llm = build_llm()
    tool_selector = build_tool_selector()

    agent = build_graph(
        router=router,
        llm=llm,
        retriever=retriever,
        tool_selector=tool_selector,
        available_tools=TOOL_SCHEMAS,
        tool_registry=TOOL_REGISTRY,
    )

    return agent

# Retrival

In [49]:
import json
from typing import Any

from langchain_core.documents import Document


def build_documents_from_dataset(
    dataset,
) -> list[Document]:

    documents = []
    seen_contents = set()

    for row in dataset:
        context = row.get("context")

        if not context:
            continue

        if isinstance(context, str):
            try:
                context = json.loads(context)
            except json.JSONDecodeError:
                continue

        if not isinstance(context, dict):
            continue

        retrieved_docs = context.get(
            "retrieved_docs",
            [],
        )

        if not isinstance(retrieved_docs, list):
            continue

        for item in retrieved_docs:

            if isinstance(item, str):
                content = item
                metadata = {}

            elif isinstance(item, dict):
                content = (
                    item.get("content")
                    or item.get("text")
                    or item.get("page_content")
                )

                metadata = {
                    key: value
                    for key, value in item.items()
                    if key not in {
                        "content",
                        "text",
                        "page_content",
                    }
                }

            else:
                continue

            if not isinstance(content, str):
                continue

            content = content.strip()

            if not content:
                continue

            if content in seen_contents:
                continue

            seen_contents.add(content)

            documents.append(
                Document(
                    page_content=content,
                    metadata=metadata,
                )
            )

    return documents

In [53]:
!pip install langchain_huggingface==1.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 31.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 138.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 120.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain_huggingface]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [55]:
from langchain_huggingface import HuggingFaceEmbeddings


def build_embedding_model():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

In [59]:
!pip install langchain_community==0.4.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [langchain_community]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [61]:
from typing import Any

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

#from .embeddings import build_embedding_model


def build_vector_store(
    documents: list[Document],
) -> FAISS:
    embedding_model = build_embedding_model()

    vector_store = FAISS.from_documents(
        documents,
        embedding_model,
    )

    return vector_store

/tmp/ipykernel_117/3413092940.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [63]:
from langchain_core.documents import Document
#from .vector_store import build_vector_store


def build_retriever(
    documents: list[Document],
    k: int = 5,
):
    vector_store = build_vector_store(
        documents
    )

    return vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": k,
        },
    )

# App

In [67]:
!pip install datasets==5.0.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.8 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.9.0
    Uninstalling fsspec-2026.9.0:
      Successfully uninstalled fsspec-2026.9.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.0
    Not uninstalling dill at /toolkit-cache/2.8.0/python3.13/kernel-libs/lib/python3.13/site-packages, outside environment /root/venv
    Can't uninstall 'dill'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [datasets]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepnote-toolkit 2.8.0 requires dill<=0.4,>=0.3.9; python_version == "3.13", but you have dill 0.4.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [69]:
from datasets import load_dataset

'''from agent.runtime import build_agent
from retrieval.documents import build_documents_from_dataset
from retrieval.retriever import build_retriever
'''

DATASET_NAME = "rescommons/Full-Ecom-Chatbot-Dataset"


def build_runtime():
    dataset = load_dataset(
        DATASET_NAME,
    )

    train_dataset = dataset["train"]

    documents = build_documents_from_dataset(
        train_dataset,
    )

    if not documents:
        raise ValueError(
            "No knowledge documents were found."
        )

    retriever = build_retriever(
        documents,
        k=5,
    )

    agent = build_agent(
        retriever=retriever,
    )

    return agent

/root/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 02\_agent\_runtime\_test\.ipynb

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from datasets import load_dataset

from retrieval.documents import build_documents_from_dataset
from retrieval.retriever import build_retriever

from agent.runtime import build_agent

In [73]:
DATASET_NAME = "rescommons/Full-Ecom-Chatbot-Dataset"

dataset = load_dataset(DATASET_NAME)
train_dataset = dataset["train"]

print(train_dataset)

Generating test split: 100%|██████████| 8818/8818 [00:00<00:00, 110153.54 examples/s]
Dataset({
    features: ['id', 'source', 'group', 'difficulty', 'system', 'history', 'prompt', 'context', 'tools', 'response_type', 'response', 'language', 'locale', 'annotator', 'quality_score', 'domain', 'intent_category', 'intent', 'sub_intent', 'capability', 'test_tier'],
    num_rows: 35213
})


In [75]:
documents = build_documents_from_dataset(train_dataset)

print(f"Number of knowledge documents: {len(documents)}")

Number of knowledge documents: 23610


In [85]:
retriever = build_retriever(
    documents=documents,
    k=5,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8598.48it/s]


KeyboardInterrupt: 

KeyboardInterrupt: 

In [89]:
!pip install -q faiss-cpu


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [95]:
#from retrieval.embeddings import build_embedding_model
import time

embedding_model = build_embedding_model()

sample_documents = documents[:500]
sample_texts = [doc.page_content for doc in sample_documents]

start = time.time()

embeddings = embedding_model.embed_documents(sample_texts)

elapsed = time.time() - start

print(f"Documents: {len(sample_texts)}")
print(f"Time: {elapsed:.2f} seconds")
print(f"Docs/sec: {len(sample_texts) / elapsed:.2f}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 33809.15it/s]
Documents: 500
Time: 211.17 seconds
Docs/sec: 2.37


In [91]:
test_documents = documents[:100]

test_retriever = build_retriever(
    documents=test_documents,
    k=5,
)

print("Test retriever built successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5350.47it/s]
Test retriever built successfully.


In [107]:
import os

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

In [111]:
import os

print("API key exists:", os.getenv("OPENAI_API_KEY"))

API key exists: YOUR_API_KEY


In [119]:
test_agent = build_agent(
    retriever=test_retriever,
)

print("Test agent built successfully.")

Test agent built successfully.


In [121]:
result = test_agent.invoke({
    "user_request": "Where is my order ORD-1001? My user ID is USER-001."
})

print("Route:", result.get("route"))
print("Selected tool:", result.get("selected_tool"))
print("Tool arguments:", result.get("tool_arguments"))
print("Tool result:", result.get("tool_result"))
print("Execution status:", result.get("execution_status"))
print("Final response:", result.get("final_response"))

OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_API_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [173]:
#from .schemas import RouteDecision, ToolCallDecision


class MockRouter:
    def invoke(self, inputs):
        user_request = inputs["user_request"].lower()

        if "cancel my order" in user_request:
            return RouteDecision(
                route="clarification",
                reason="The user wants to cancel an order but has not provided the order ID.",
                missing_information=["order_id"],
            )

        if "fraud" in user_request or "stolen" in user_request:
            return RouteDecision(
                route="human",
                reason="The request involves a potentially sensitive or fraudulent situation.",
                missing_information=[],
            )

        if "order" in user_request:
            return RouteDecision(
                route="tool",
                reason="The user is asking for dynamic order information.",
                missing_information=[],
            )

        if "return policy" in user_request or "return" in user_request:
            return RouteDecision(
                route="rag",
                reason="The user is asking about static support knowledge.",
                missing_information=[],
            )

        return RouteDecision(
            route="direct",
            reason="No external information or tool is required.",
            missing_information=[],
        )

class MockToolSelector:
    def invoke(self, inputs):
        return ToolCallDecision(
            tool_name="get_order_status",
            arguments={
                "order_id": "ORD-1001",
                "user_id": "USER-001",
            },
            reason="The request asks for the status of an order.",
        )


class MockLLM:
    def invoke(self, inputs):

        # Clarification / Human prompts are passed as strings
        if isinstance(inputs, str):

            if "must be escalated to a human support agent" in inputs:
                return type(
                    "MockResponse",
                    (),
                    {
                        "content": (
                            "Your request will be handled by a human support agent."
                        )
                    },
                )()

            if "Ask the user for the minimum missing information" in inputs:
                return type(
                    "MockResponse",
                    (),
                    {
                        "content": "Could you please provide the order ID?"
                    },
                )()

            return type(
                "MockResponse",
                (),
                {
                    "content": "Mock response."
                },
            )()

        tool_result = inputs.get("tool_result")
        retrieved_documents = inputs.get("retrieved_documents", [])

        if tool_result:
            return type(
                "MockResponse",
                (),
                {
                    "content": (
                        f"Your order information is: {tool_result}"
                    )
                },
            )()

        if retrieved_documents:
            first_doc = retrieved_documents[0]["content"]

            return type(
                "MockResponse",
                (),
                {
                    "content": (
                        f"Answer generated using retrieved knowledge: "
                        f"{first_doc}"
                    )
                },
            )()

        return type(
            "MockResponse",
            (),
            {
                "content": "This is a mock direct response."
            },
        )()

In [175]:
mock_router = MockRouter()
mock_tool_selector = MockToolSelector()
mock_llm = MockLLM()

In [177]:
test_agent = build_graph(
    router=MockRouter(),
    llm=MockLLM(),
    retriever=test_retriever,
    tool_selector=MockToolSelector(),
    available_tools=TOOL_SCHEMAS,
    tool_registry=TOOL_REGISTRY,
)

print("Test agent rebuilt successfully.")

Test agent rebuilt successfully.


In [129]:
result = test_agent.invoke(
    {
        "user_request": "Where is my order ORD-1001? My user ID is USER-001."
    }
)


print("Route:", result.get("route"))
print("Selected tool:", result.get("selected_tool"))
print("Tool arguments:", result.get("tool_arguments"))
print("Validation:", result.get("validation_status"))
print("Guardrail:", result.get("guardrail_result"))
print("Execution:", result.get("execution_status"))
print("Tool result:", result.get("tool_result"))
print("Final response:", result.get("final_response"))

Route: tool
Selected tool: get_order_status
Tool arguments: {'order_id': 'ORD-1001', 'user_id': 'USER-001'}
Validation: valid
Guardrail: allowed
Execution: successful
Tool result: {'success': True, 'order_id': 'ORD-1001', 'status': 'shipped'}
Final response: Your order information is: {'success': True, 'order_id': 'ORD-1001', 'status': 'shipped'}


In [143]:
result = test_agent.invoke(
    {
        "user_request": "What is the return policy?"
    }
)

print("Route:", result.get("route"))
print("Retrieved documents:", len(result.get("retrieved_documents", [])))
print("Final response:", result.get("final_response"))
print("Output guard:", result.get("output_guard_result"))

Route: rag
Retrieved documents: 5
Final response: Answer generated using retrieved knowledge: The Psychology of Money by Morgan Housel. Type: finance book. Key feature: finance. Price: ₹399. Domain: books_media.
Output guard: passed


In [145]:
result = test_agent.invoke(
    {
        "user_request": "Hello, can you help me?"
    }
)

print("Route:", result.get("route"))
print("Final response:", result.get("final_response"))
print("Output guard:", result.get("output_guard_result"))

Route: direct
Final response: This is a mock direct response.
Output guard: passed


In [163]:
result = test_agent.invoke(
    {
        "user_request": "I want to cancel my order."
    }
)

print("Route:", result.get("route"))
print("Missing information:", result.get("missing_information"))
print("Final response:", result.get("final_response"))
print("Output guard:", result.get("output_guard_result"))

Route: clarification
Missing information: ['order_id']
Final response: Could you please provide the order ID?
Output guard: passed


In [155]:
router = MockRouter()

decision = router.invoke({
    "user_request": "I want to cancel my order."
})

print(type(decision))
print(decision)
print(type(decision.missing_information))
print(decision.missing_information)

<class '__main__.RouteDecision'>
route='clarification' reason='The user wants to cancel an order but has not provided the order ID.' missing_information=['order_id']
<class 'list'>
['order_id']


In [179]:
result = test_agent.invoke(
    {
        "user_request": "I think my card was stolen and there may be fraud."
    }
)

print("Route:", result.get("route"))
print("Escalation required:", result.get("escalation_required"))
print("Escalation reason:", result.get("escalation_reason"))
print("Final response:", result.get("final_response"))
print("Output guard:", result.get("output_guard_result"))

Route: human
Escalation required: True
Escalation reason: The request requires human assistance.
Final response: Your request will be handled by a human support agent.
Output guard: passed


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=81ed7788-49aa-4fff-b06f-3a6ebbf89c8e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>